In [0]:
import json
import random
from datetime import datetime, timedelta, timezone

spark.conf.set("spark.sql.session.timeZone", "UTC")

SEED = 42
random.seed(SEED)

BASE = "/Volumes/training_lab_classic/cold_chain/volume1"

SHIPMENTS_PATH = f"{BASE}/landing/shipments"
PRODUCTS_PATH = f"{BASE}/landing/product_limits"
DELIVERIES_PATH = f"{BASE}/landing/delivery_events"

SENSOR_LANDING = f"{BASE}/landing/sensor_events"
SENSOR_STAGING = f"{BASE}/staging/sensor_events"

dbutils.fs.mkdirs(SENSOR_STAGING)

print("Data generator initialized")

In [0]:
products = [
    ("P001", 2.0, 8.0),
    ("P002", 3.0, 7.0),
    ("P003", 1.0, 5.0)
]

products_df = spark.createDataFrame(
    products,
    """
    product_id STRING,
    min_temperature DOUBLE,
    max_temperature DOUBLE
    """
)

display(products_df)

In [0]:
shipments = [
    (
        "SH001", "P001", "V001",
        "Pune", "Mumbai",
        "2026-10-07T08:00:00Z",
        "2026-10-07T12:00:00Z"
    ),
    (
        "SH002", "P002", "V002",
        "Mumbai", "Pune",
        "2026-10-07T08:30:00Z",
        "2026-10-07T13:00:00Z"
    ),
    (
        "SH003", "P003", "V003",
        "Pune", "Bengaluru",
        "2026-10-07T07:00:00Z",
        "2026-10-07T15:00:00Z"
    ),
    (
        "SH004", "P001", "V004",
        "Delhi", "Pune",
        "2026-10-07T06:00:00Z",
        "2026-10-07T14:00:00Z"
    ),
    (
        "SH005", "P002", "V005",
        "Hyderabad", "Chennai",
        "2026-10-07T09:00:00Z",
        "2026-10-07T16:00:00Z"
    )
]

shipments_df = spark.createDataFrame(
    shipments,
    """
    shipment_id STRING,
    product_id STRING,
    vehicle_id STRING,
    origin STRING,
    destination STRING,
    dispatch_time STRING,
    expected_arrival STRING
    """
)

display(shipments_df)


In [0]:
delivery_events = [
    # 30 min late
    (
        "D001",
        "SH001",
        "DELIVERED",
        "2026-10-07T12:30:00Z"
    ),

    # 10 min early
    (
        "D002",
        "SH002",
        "DELIVERED",
        "2026-10-07T12:50:00Z"
    ),

    # SH003 deliberately not delivered yet

    # exactly on time
    (
        "D004",
        "SH004",
        "DELIVERED",
        "2026-10-07T14:00:00Z"
    ),

    # 60 min late
    (
        "D005",
        "SH005",
        "DELIVERED",
        "2026-10-07T17:00:00Z"
    )
]

deliveries_df = spark.createDataFrame(
    delivery_events,
    """
    event_id STRING,
    shipment_id STRING,
    status STRING,
    event_time STRING
    """
)

display(deliveries_df)

In [0]:
sensor_events = [

    # ---------- SH001 / P001 / allowed 2-8 ----------

    ("E001", "SH001", "S001", "2026-10-07T08:00:00Z", "5.0"),
    ("E002", "SH001", "S001", "2026-10-07T08:05:00Z", "6.0"),

    # Upper boundary. Should still be accepted.
    ("E003", "SH001", "S001", "2026-10-07T08:10:00Z", "8.0"),

    # Temperature exception
    ("E004", "SH001", "S001", "2026-10-07T08:15:00Z", "11.0"),


    # ---------- SH002 / P002 / allowed 3-7 ----------

    ("E005", "SH002", "S002", "2026-10-07T08:30:00Z", "4.0"),
    ("E006", "SH002", "S002", "2026-10-07T08:35:00Z", "5.0"),

    # Lower boundary. Accepted.
    ("E007", "SH002", "S002", "2026-10-07T08:40:00Z", "3.0"),

    # High exception
    ("E008", "SH002", "S002", "2026-10-07T08:45:00Z", "9.0"),


    # ---------- SH003 / P003 / allowed 1-5 ----------

    ("E009", "SH003", "S003", "2026-10-07T07:00:00Z", "3.0"),
    ("E010", "SH003", "S003", "2026-10-07T07:05:00Z", "4.0"),

    # Low exception
    ("E011", "SH003", "S003", "2026-10-07T07:10:00Z", "-1.0"),


    # ---------- SH004 / P001 ----------

    ("E012", "SH004", "S004", "2026-10-07T06:00:00Z", "4.0"),
    ("E013", "SH004", "S004", "2026-10-07T06:05:00Z", "5.0"),
    ("E014", "SH004", "S004", "2026-10-07T06:10:00Z", "6.0"),


    # ---------- SH005 / P002 ----------

    ("E015", "SH005", "S005", "2026-10-07T09:00:00Z", "4.5"),
    ("E016", "SH005", "S005", "2026-10-07T09:05:00Z", "5.5"),


    # ---------- BAD RECORDS ----------

    # Missing event ID
    (None, "SH005", "S005", "2026-10-07T09:10:00Z", "6.0"),

    # Unknown shipment
    ("E018", "SH999", "S999", "2026-10-07T09:15:00Z", "5.0"),

    # Nonnumeric temperature
    ("E019", "SH005", "S005", "2026-10-07T09:20:00Z", "hot"),

    # Duplicate of E015
    ("E015", "SH005", "S005", "2026-10-07T09:00:00Z", "4.5")
]


In [0]:
len(sensor_events)


In [0]:
batch_1 = sensor_events[:8]
batch_2 = sensor_events[8:14]
batch_3 = sensor_events[14:]

In [0]:
print("Batch 1:", len(batch_1))
print("Batch 2:", len(batch_2))
print("Batch 3:", len(batch_3))
print("Total:", len(batch_1) + len(batch_2) + len(batch_3))

In [0]:
def write_sensor_batch(filename, records):

    output = []

    for row in records:

        record = {
            "event_id": row[0],
            "shipment_id": row[1],
            "sensor_id": row[2],
            "event_time": row[3],
            "temperature_c": row[4]
        }

        output.append(json.dumps(record))

    path = f"{SENSOR_STAGING}/{filename}"

    dbutils.fs.put(
        path,
        "\n".join(output) + "\n",
        overwrite=False
    )

    print(
        f"Created {filename} with {len(records)} records"
    )

In [0]:
write_sensor_batch(
    "batch_001.json",
    batch_1
)

write_sensor_batch(
    "batch_002.json",
    batch_2
)

write_sensor_batch(
    "batch_003.json",
    batch_3
)

In [0]:
display(
    dbutils.fs.ls(SENSOR_STAGING)
)


In [0]:
overwrite=False

In [0]:
(
    products_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(PRODUCTS_PATH)
)

In [0]:
(
    shipments_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(SHIPMENTS_PATH)
)


In [0]:
(
    deliveries_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .json(DELIVERIES_PATH)
)

In [0]:
print("Products")
display(dbutils.fs.ls(PRODUCTS_PATH))

print("Shipments")
display(dbutils.fs.ls)